In [1]:
import os
import tensorflow as tf
from pathlib import Path

print("TensorFlow Version:", tf.__version__)
print("Current Working Directory:", os.getcwd())

TensorFlow Version: 2.12.0
Current Working Directory: C:\Users\Ramadan\Documents\ML_Projects\Kidney-Disease-Classification-Deep-Learning-Project


In [2]:
import zipfile
import gdown

# إعداد المسارات بناءً على هيكلة مشروعك
os.makedirs("artifacts/data_ingestion", exist_ok=True)

dataset_url = "https://drive.google.com/file/d/1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3/view?usp=sharing"
zip_download_dir = "artifacts/data_ingestion/data.zip"

print(f"Downloading data from {dataset_url}...")
file_id = dataset_url.split("/")[-2]
prefix = 'https://drive.google.com/uc?/export=download&id='
gdown.download(prefix + file_id, zip_download_dir, quiet=False)

# فك ضغط الملفات
unzip_path = "artifacts/data_ingestion"
os.makedirs(unzip_path, exist_ok=True)
with zipfile.ZipFile(zip_download_dir, 'r') as zip_ref:
    zip_ref.extractall(unzip_path)

print("Data Ingestion Completed Successfully!")

Downloading...
From (original): https://drive.google.com/uc?/export=download&id=1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3
From (redirected): https://drive.google.com/uc?%2Fexport=download&id=1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3&confirm=t&uuid=324edef7-88bd-4354-a5b0-8f2fa93b7ab3
To: C:\Users\Ramadan\Documents\ML_Projects\Kidney-Disease-Classification-Deep-Learning-Project\artifacts\data_ingestion\data.zip
100%|██████████| 57.7M/57.7M [07:45<00:00, 124kB/s] 


Data Ingestion Completed Successfully!


In [4]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Flatten, Dense, Input
from tensorflow.keras.models import Model

# إعدادات المعمارية
IMAGE_SIZE = [224, 224, 3]
CLASSES = 2

# تحميل نموذج VGG16 الأساسي بدون طبقات التصنيف العلوية
base_model = VGG16(input_shape=IMAGE_SIZE, weights='imagenet', include_top=False)

# تجميد طبقات النموذج الأساسي
base_model.trainable = False

# إضافة طبقات مخصصة لتصنيف الكلى
x = Flatten()(base_model.output)
prediction = Dense(CLASSES, activation='softmax')(x)

full_model = Model(inputs=base_model.input, outputs=prediction)

full_model.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
    metrics=['accuracy']
)

full_model.summary()

# حفظ النموذج الأساسي المحدث
os.makedirs("artifacts/prepare_base_model", exist_ok=True)
full_model.save("artifacts/prepare_base_model/base_model_updated.h5")
print("Base Model Prepared and Saved Successfully!")

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0     

In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# تحديد مجلد البيانات (تأكد من اسم المجلد المفكوك الناتج داخل data_ingestion مثل Chest-CT-Scan-data أو Kidney-Data)
# يمكنك تعديل المسار أدناه ليطابق المجلد الفعلي الناتج عن فك الـ zip
training_data_dir = "artifacts/data_ingestion" # أو المسار الفرعي المخصص للصور إذا وجد

# إعداد مولد البيانات مع Data Augmentation
datagen_kwargs = dict(rescale=1./255, validation_split=0.20)

dataflow_kwargs = dict(
    target_size=(224, 224),
    batch_size=16,
    class_mode='categorical'
)

valid_datagen = ImageDataGenerator(**datagen_kwargs)
valid_generator = valid_datagen.flow_from_directory(
    directory=training_data_dir,
    subset="validation",
    shuffle=False,
    **dataflow_kwargs
)

train_datagen = ImageDataGenerator(**datagen_kwargs)
train_generator = train_datagen.flow_from_directory(
    directory=training_data_dir,
    subset="training",
    shuffle=True,
    **dataflow_kwargs
)

# تدريب النموذج (EPOCHS = 1 للتجربة السريعة ويمكن زيادتها)
history = full_model.fit(
    train_generator,
    epochs=1,
    validation_data=valid_generator,
    steps_per_epoch=len(train_generator),
    validation_steps=len(valid_generator)
)

# حفظ النموذج المدرب نهائياً
os.makedirs("artifacts/training", exist_ok=True)
full_model.save("artifacts/training/model.h5")
print("Model Training Completed and Saved!")

Found 93 images belonging to 1 classes.
Found 372 images belonging to 1 classes.
24/24 [==============================] - 99s 4s/step - loss: 34.2853 - accuracy: 0.4812 - val_loss: 51.9945 - val_accuracy: 1.0000
Model Training Completed and Saved!


In [15]:
import os
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

class PredictionPipeline:
    def __init__(self, filename):
        self.filename = filename

    def predict(self):
        # تحميل النموذج المدرب من المسار الذي حفظناه فيه
        model_path = os.path.join("artifacts", "training", "model.h5")
        model = load_model(model_path)

        # قراءة الصورة ومعالجتها بالحجم المطلوب (المتوافق مع VGG16 وهو 224x224)
        imagename = self.filename
        test_image = image.load_img(imagename, target_size=(224, 224))
        test_image = image.img_to_array(test_image)
        test_image = np.expand_dims(test_image, axis=0)
        test_image = test_image / 255.0  # تطبيع القيم (Normalization)

        # إجراء التنبؤ
        result = np.argmax(model.predict(test_image), axis=1)
        print(f"Prediction Array Result: {result}")

        # بناءً على فئات البيانات (مثلاً: Cysts, Tumor, Stone, Normal إلخ)
        # سنعرض النتيجة بناءً على الفهرس (Index) المتوقع
        if result[0] == 1:
            prediction = "Tumor / Abnormal (ورم أو غير طبيعي)"
            return [{"image": prediction}]
        else:
            prediction = "Normal (طبيعي)"
            return [{"image": prediction}]

print("تم تعريف PredictionPipeline بنجاح!")

تم تعريف PredictionPipeline بنجاح!


In [21]:
import os
from cnnClassifier.pipeline.prediction import PredictionPipeline # أو الكود الذي عرفناه سابقاً

# اختيار مسار صورة حقيقية من مجلد Normal
test_image_path = r"C:\Users\Ramadan\Downloads\images.jpg"

if os.path.exists(test_image_path):
    predictor = PredictionPipeline(test_image_path)
    prediction_result = predictor.predict()
    print("نتيجة التنبؤ للنموذج:", prediction_result)
else:
    print(f"مسار الصورة غير موجود، تأكدي من المسار: {test_image_path}")

1/1 [==============================] - 1s 617ms/step
[0]
نتيجة التنبؤ للنموذج: [{'image': 'Normal'}]
